In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(r"C:\Users\luis\Documents\TFG - Data-Centric AI")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd
from IPython.display import display

from src.phase3.config import PHASE3_DARKNESS_THRESHOLD
from utils.thresholds.quality import (
    ensure_quality_metric,
    evaluate_quality_thresholds_against_manual_labels,
    export_quality_review_images,
    get_quality_filter_spec,
    recommend_quality_threshold,
    sample_images_for_quality_threshold_review,
)

FILTER_NAME = "uniformity"
METADATA_PATH = PROJECT_ROOT / "data" / "phase2" / "phase2_train.csv"
IMAGES_DIR = PROJECT_ROOT / "data" / "phase2" / "frames"

REPORTS_DIR = PROJECT_ROOT / "reports" / f"{FILTER_NAME}_threshold_selection"
REVIEW_IMAGES_DIR = REPORTS_DIR / "review_images"
REVIEW_CSV_PATH = REPORTS_DIR / "manual_quality_review.csv"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

spec = get_quality_filter_spec(FILTER_NAME)
METRIC_COLUMN = spec["metric_column"]
THRESHOLD_CANDIDATES = spec["thresholds"]
N_REVIEW_IMAGES = 100
N_REVIEW_IMAGES_PER_THRESHOLD = N_REVIEW_IMAGES // len(THRESHOLD_CANDIDATES)

print("FILTER_NAME:", FILTER_NAME)
print("METADATA_PATH:", METADATA_PATH)
print("IMAGES_DIR:", IMAGES_DIR)
print("THRESHOLD_CANDIDATES:", THRESHOLD_CANDIDATES)
print("DARKNESS_PREFILTER_THRESHOLD:", PHASE3_DARKNESS_THRESHOLD)

FILTER_NAME: uniformity
METADATA_PATH: C:\Users\luis\Documents\TFG - Data-Centric AI\data\phase2\phase2_train.csv
IMAGES_DIR: C:\Users\luis\Documents\TFG - Data-Centric AI\data\phase2\frames
THRESHOLD_CANDIDATES: [5.75, 6.0, 6.25, 6.5, 6.75]
DARKNESS_PREFILTER_THRESHOLD: 50.0


In [3]:
df = pd.read_csv(METADATA_PATH)
df = ensure_quality_metric(
    dataframe=df,
    filter_name="darkness",
    images_dir=IMAGES_DIR,
)

images_before_darkness = len(df)
df = df[df["brightness_v_mean"] >= PHASE3_DARKNESS_THRESHOLD].copy()

df = ensure_quality_metric(
    dataframe=df,
    filter_name=FILTER_NAME,
    images_dir=IMAGES_DIR,
)

print("Images before darkness prefilter:", images_before_darkness)
print("Images after darkness prefilter:", len(df))
display(df[["filename", "brightness_v_mean", METRIC_COLUMN]].head())
display(df[METRIC_COLUMN].describe())

Images before darkness prefilter: 7978
Images after darkness prefilter: 7907


,filename,brightness_v_mean,uniformity_entropy
0,20241204_134604_R0_F0_S1_8f99b423ac93fa5b.jpg,116.906393,7.079925
1,20241213_102820_R0_F0_S1_34b16ba5d8336515.jpg,84.247481,7.057979
2,20241213_102821_R2_F0_S1_34b16ba5d8336515.jpg,76.631638,6.373658
3,20241213_103001_R2_F0_S8_34b16ba5d8336515.jpg,131.618293,6.833271
4,20241213_103008_R2_F0_S9_34b16ba5d8336515.jpg,98.607903,7.445012


count    7907.000000
mean        7.008322
std         0.312093
min         4.798007
25%         6.852412
50%         7.057137
75%         7.225987
max         7.642709
Name: uniformity_entropy, dtype: float64

In [4]:
review_df = sample_images_for_quality_threshold_review(
    dataframe=df,
    filter_name=FILTER_NAME,
    thresholds=THRESHOLD_CANDIDATES,
    n_per_threshold=N_REVIEW_IMAGES_PER_THRESHOLD,
)

review_df = export_quality_review_images(
    review_df=review_df,
    images_dir=IMAGES_DIR,
    output_dir=REVIEW_IMAGES_DIR,
    image_size=(320, 180),
)

review_df.to_csv(REVIEW_CSV_PATH, index=False)

print("CSV to label:", REVIEW_CSV_PATH)
print("Images exported to:", REVIEW_IMAGES_DIR)
display(review_df.head())

CSV to label: C:\Users\luis\Documents\TFG - Data-Centric AI\reports\uniformity_threshold_selection\manual_quality_review.csv
Images exported to: C:\Users\luis\Documents\TFG - Data-Centric AI\reports\uniformity_threshold_selection\review_images


,patient_id,day,R,F,hour,histology,filename,video_filename,elapsed_seconds,crop_x,...,bbox_area_ratio,brightness_v_mean,uniformity_entropy,filter_name,threshold,predicted_discard,distance_to_threshold,manual_label,manual_comment,review_image_path
0,4483322,20241209,R5,F0,NaN,Sessile_serrated_adenoma,20241209_093253_R5_F0_S2_f39c8981ba688cb1_3.jpg,20241209_093210_R5_f39c8981ba688cb1.mp4,43.0,291,...,0.402412,62.030067,5.744425,uniformity,5.75,True,0.005575,,,C:\Users\luis\Documents\TFG - Data-Centric AI\...
1,5463981,20250414,R7,F0,NaN,Hyperplastic,20250414_135953_R7_F0_S2_90985f258ff468af_1.jpg,20250414_135938_R7_90985f258ff468af.mp4,15.0,411,...,0.746153,141.634817,5.736221,uniformity,5.75,True,0.013779,,,C:\Users\luis\Documents\TFG - Data-Centric AI\...
2,70653713,20250207,R3,F0,NaN,Hyperplastic,20250207_111715_R3_F0_S7_ea1358d540cd4672_1.jpg,20250207_111621_R3_ea1358d540cd4672.mp4,54.0,323,...,0.812191,181.549356,5.717514,uniformity,5.75,True,0.032486,,,C:\Users\luis\Documents\TFG - Data-Centric AI\...
3,4775750,20250306,R12,F0,NaN,Adenoma,20250306_102115_R12_F0_S1_55bbbf6391713ff2_1.jpg,20250306_102113_R12_55bbbf6391713ff2.mp4,2.0,353,...,0.024020,174.659253,5.690740,uniformity,5.75,True,0.059260,,,C:\Users\luis\Documents\TFG - Data-Centric AI\...
4,4483322,20241209,R3,F0,NaN,Adenoma,20241209_092430_R3_F0_S5_f39c8981ba688cb1_1.jpg,20241209_092358_R3_f39c8981ba688cb1.mp4,32.0,291,...,0.815524,83.673767,5.688625,uniformity,5.75,True,0.061375,,,C:\Users\luis\Documents\TFG - Data-Centric AI\...


In [5]:
labeled_df = pd.read_csv(REVIEW_CSV_PATH, sep=None, engine="python")
labeled_df["manual_label"] = (
    labeled_df["manual_label"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

valid_labels = {"discard", "valuable", "uncertain"}
invalid_labels = set(labeled_df["manual_label"].unique()) - valid_labels

if invalid_labels:
    raise ValueError(f"Invalid labels: {invalid_labels}")

label_counts = (
    labeled_df["manual_label"]
    .value_counts()
    .rename_axis("manual_label")
    .reset_index(name="count")
)

display(label_counts)

,manual_label,count
0,discard,51
1,valuable,49


In [6]:
manual_evaluation_df = evaluate_quality_thresholds_against_manual_labels(
    labeled_df=labeled_df,
    filter_name=FILTER_NAME,
    thresholds=THRESHOLD_CANDIDATES,
    positive_labels=("discard",),
    negative_labels=("valuable",),
)

manual_evaluation_df = manual_evaluation_df.sort_values(
    ["precision", "recall", "false_positive"],
    ascending=[False, False, True],
).reset_index(drop=True)

display(manual_evaluation_df)

,filter_name,threshold,labelled_images,true_positive,false_positive,false_negative,true_negative,precision,recall
0,uniformity,5.75,100,7,1,44,48,0.875000,0.137255
1,uniformity,6.25,100,28,10,23,39,0.736842,0.549020
2,uniformity,6.00,100,15,7,36,42,0.681818,0.294118
3,uniformity,6.50,100,33,23,18,26,0.589286,0.647059
4,uniformity,6.75,100,36,35,15,14,0.507042,0.705882


In [7]:
recommended_df = recommend_quality_threshold(
    evaluation_df=manual_evaluation_df,
    min_precision=0.90,
)

display(recommended_df)

selected = recommended_df.iloc[0]
FINAL_THRESHOLD = float(selected["threshold"])

print(f"Selected {FILTER_NAME} threshold: {FINAL_THRESHOLD:g}")

,filter_name,threshold,labelled_images,true_positive,false_positive,false_negative,true_negative,precision,recall
0,uniformity,5.75,100,7,1,44,48,0.875000,0.137255
1,uniformity,6.25,100,28,10,23,39,0.736842,0.549020
2,uniformity,6.00,100,15,7,36,42,0.681818,0.294118
3,uniformity,6.50,100,33,23,18,26,0.589286,0.647059
4,uniformity,6.75,100,36,35,15,14,0.507042,0.705882


Selected uniformity threshold: 5.75
